<a href="https://colab.research.google.com/github/jeraemiusshannonj/MLOps-Inference-Optimization-ONNX/blob/main/inference_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install onnx onnxscript tensorrt -q
!pip install tensorrt onnx -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 20.6 MB/s eta 0:00:00


In [15]:
import torch
import torchvision.models as models

# Verifying CUDA availability
if not torch.cuda.is_available():
    raise SystemError("GPU not found. Ensure Runtime > Change runtime type is set to T4 GPU.")

device = torch.device('cuda')


model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).eval().to(device)
dummy_input = torch.randn(1, 3, 224, 224, device=device)

# Tracing and exporting computational graph to ONNX
torch.onnx.export(
    model,
    dummy_input,
    "resnet18.onnx",
    export_params=True,
    opset_version=18,
    input_names=['input'],
    output_names=['output']
)

print("SUCCESS: resnet18.onnx exported with CUDA is enabled")

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
SUCCESS: resnet18.onnx exported with CUDA is enabled


In [17]:
import torch
import torchvision.models as models

# checking if GPU is active
assert torch.cuda.is_available(), "GPU not found. Set Runtime > Change runtime type to T4 GPU."
device = torch.device('cuda')
batch_size = 32
# loading PyTorch ResNet-18 Model (FP32 Baseline)
model_fp32 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device).eval()
dummy_input_fp32 = torch.randn(batch_size, 3, 224, 224, device=device)

# Warming up the GPU
for _ in range(50):
    _ = model_fp32(dummy_input_fp32)
torch.cuda.synchronize()

# Measuring FP32 Latency
starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
repetitions = 200
timings_fp32 = []

with torch.no_grad():
    for _ in range(repetitions):
        starter.record()
        _ = model_fp32(dummy_input_fp32)
        ender.record()
        torch.cuda.synchronize()
        timings_fp32.append(starter.elapsed_time(ender))

fp32_latency = sum(timings_fp32) / repetitions
fp32_qps = (batch_size * 1000.0) / fp32_latency

# Casting Model and Tensor to FP16 (Tensor Core Precision)
model_fp16 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device).half().eval()
dummy_input_fp16 = dummy_input_fp32.half()

# Warming up FP16
for _ in range(50):
    _ = model_fp16(dummy_input_fp16)
torch.cuda.synchronize()

# Measuring FP16 Latency
timings_fp16 = []
with torch.no_grad():
    for _ in range(repetitions):
        starter.record()
        _ = model_fp16(dummy_input_fp16)
        ender.record()
        torch.cuda.synchronize()
        timings_fp16.append(starter.elapsed_time(ender))

fp16_latency = sum(timings_fp16) / repetitions
fp16_qps = (batch_size * 1000.0) / fp16_latency

# 4. Calculating Final Metrics
speedup = fp32_latency / fp16_latency
latency_reduction = ((fp32_latency - fp16_latency) / fp32_latency) * 100

print(f"***--------FINAL RESULTS--------***")
print(f"Batch Size: {batch_size}")
print(f"FP32 Baseline Latency : {fp32_latency:.2f} ms | Throughput: {fp32_qps:.1f} QPS")
print(f"FP16 Accelerated Latency: {fp16_latency:.2f} ms | Throughput: {fp16_qps:.1f} QPS")
print(f"Latency Reduction     : {latency_reduction:.1f}%")
print(f"Speedup Factor        : {speedup:.2f}x")

***--------FINAL RESULTS--------***
Batch Size: 32
FP32 Baseline Latency : 33.74 ms | Throughput: 948.5 QPS
FP16 Accelerated Latency: 16.21 ms | Throughput: 1973.9 QPS
Latency Reduction     : 51.9%
Speedup Factor        : 2.08x
